In [ ]:
from kafka import KafkaProducer
import boto3
import json
import requests
import time

topic = "air_quality_stream"

bucket = "openaq-project-dm"
prefix = "raw/latest/"

API_KEY = "MY_API_KEY"

headers = {
    "X-API-Key": API_KEY
}

# -------------------------
# INIT SERVICES
# -------------------------

producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

s3 = boto3.client("s3")

# -------------------------
# READ LATEST FILES
# -------------------------

response = s3.list_objects_v2(
    Bucket=bucket,
    Prefix=prefix
)

files = [
    obj["Key"]
    for obj in response.get("Contents", [])
    if obj["Key"].endswith(".json")
]

print("Latest files:", len(files))

location_ids = set()

for file in files:

    obj = s3.get_object(Bucket=bucket, Key=file)

    data = json.loads(obj["Body"].read())

    for r in data.get("results", []):

        if "locationsId" in r:
            location_ids.add(r["locationsId"])

print("Unique locations:", len(location_ids))

for loc_id in location_ids:

    url = f"https://api.openaq.org/v3/sensors/{loc_id}"

    print("Calling:", url)

    try:

        r = requests.get(url, headers=headers)

        if r.status_code == 429:
            print("Rate limit hit")
            time.sleep(10)
            continue

        if r.status_code != 200:
            print("API error:", r.text)
            continue

        data = r.json()

        for record in data.get("results", []):

            producer.send(topic, record)

        print("Records sent:", len(data.get("results", [])))

        time.sleep(1)

    except Exception as e:

        print("Producer error:", e)

producer.flush()

print("Sensor producer finished")

Latest files: 10
Unique locations: 1120
Calling: https://api.openaq.org/v3/sensors/13
Records sent: 1
Calling: https://api.openaq.org/v3/sensors/17
Records sent: 1
Calling: https://api.openaq.org/v3/sensors/19
Records sent: 1
Calling: https://api.openaq.org/v3/sensors/21
Records sent: 1
Calling: https://api.openaq.org/v3/sensors/23
Records sent: 1
Calling: https://api.openaq.org/v3/sensors/25
Records sent: 1
Calling: https://api.openaq.org/v3/sensors/26
Records sent: 1
Calling: https://api.openaq.org/v3/sensors/27
Records sent: 1
Calling: https://api.openaq.org/v3/sensors/28
Records sent: 1
Calling: https://api.openaq.org/v3/sensors/30
Records sent: 1
Calling: https://api.openaq.org/v3/sensors/31
Records sent: 1
Calling: https://api.openaq.org/v3/sensors/33
Records sent: 1
Calling: https://api.openaq.org/v3/sensors/38
Records sent: 1
Calling: https://api.openaq.org/v3/sensors/39
API error: {"detail":"Sensor not found"}
Calling: https://api.openaq.org/v3/sensors/41
Records sent: 1
Calli

In [ ]:
from kafka import KafkaConsumer
import boto3
import json
import datetime

topic = "air_quality_stream"

bucket = "openaq-project-dm"
prefix = "raw/Sensors/"

batch_size = 500

consumer = KafkaConsumer(
    topic,
    bootstrap_servers="localhost:9092",
    value_deserializer=lambda x: json.loads(x.decode("utf-8")),
    auto_offset_reset="earliest",
    enable_auto_commit=True
)

s3 = boto3.client("s3")

print("Sensor consumer started")

buffer = []
seen = set()

for message in consumer:

    record = message.value

    try:

        unique_key = f"{record['id']}_{record['parameter']}"

        if unique_key in seen:
            continue

        seen.add(unique_key)

        buffer.append(record)

        print("Buffered:", unique_key)

    except Exception as e:
        print("Parse error:", e)
        continue


    if len(buffer) >= batch_size:

        timestamp = datetime.datetime.utcnow().strftime("%Y-%m-%d_%H:%M:%S")

        key = f"{prefix}{timestamp}.json"

        try:

            s3.put_object(
                Bucket=bucket,
                Key=key,
                Body=json.dumps({"results": buffer})
            )

            print("Uploaded:", key)

            buffer = []

        except Exception as e:

            print("S3 error:", e)

ERROR! Session/line number was not unique in database. History logging moved to new session 35
Sensor consumer started
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'
Parse error: 'id'

C:\Users\admin\AppData\Local\Temp\ipykernel_604\3324542310.py:68: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.datetime.utcnow().strftime("%Y-%m-%d_%H:%M:%S")


Uploaded: raw/Sensors/2026-03-09_17:26:34.json
Buffered: 1174_{'id': 10, 'name': 'o3', 'units': 'ppm', 'displayName': 'O₃'}
Buffered: 1175_{'id': 2, 'name': 'pm25', 'units': 'µg/m³', 'displayName': 'PM2.5'}
Buffered: 1176_{'id': 2, 'name': 'pm25', 'units': 'µg/m³', 'displayName': 'PM2.5'}
Buffered: 1177_{'id': 2, 'name': 'pm25', 'units': 'µg/m³', 'displayName': 'PM2.5'}
Buffered: 1178_{'id': 1, 'name': 'pm10', 'units': 'µg/m³', 'displayName': 'PM10'}
Buffered: 1184_{'id': 4, 'name': 'co', 'units': 'µg/m³', 'displayName': 'CO mass'}
Buffered: 1185_{'id': 2, 'name': 'pm25', 'units': 'µg/m³', 'displayName': 'PM2.5'}
Buffered: 1186_{'id': 2, 'name': 'pm25', 'units': 'µg/m³', 'displayName': 'PM2.5'}
Buffered: 1187_{'id': 10, 'name': 'o3', 'units': 'ppm', 'displayName': 'O₃'}
Buffered: 1188_{'id': 2, 'name': 'pm25', 'units': 'µg/m³', 'displayName': 'PM2.5'}
Buffered: 1193_{'id': 2, 'name': 'pm25', 'units': 'µg/m³', 'displayName': 'PM2.5'}
Buffered: 1194_{'id': 6, 'name': 'so2', 'units': 'µg/